# Method 2: Smaller Theta Grid Spacing

Reduce shell grid spacing to increase theta coverage and make the LF theta surface denser.

This notebook follows the same CNP and MF-GP pattern as the base shell-theta experiment, but uses a variation-specific shell config and artifact version.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "prepare_resum_data.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not find the XLZD repo root from the current working directory.")

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT / "src" / "run_cnp") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src" / "run_cnp"))
if str(REPO_ROOT / "src" / "run_mfgp") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src" / "run_mfgp"))

from cnp_clean_pipeline import load_runtime_config, predict_cnp, train_cnp
from mfgp_clean_pipeline import run_mfgp_transform_suite

VARIATION = "method2_smaller_grid"
PREP_CONFIG_PATH = REPO_ROOT / "xlzd_shell_theta" / "variations" / "config" / "method2_smaller_grid.json"
CONFIG_PATH = REPO_ROOT / "xlzd_shell_theta" / "variations" / "settings" / "settings_method2_smaller_grid_minibatch.yaml"
VALIDATION_CONFIG_PATH = REPO_ROOT / "xlzd_shell_theta" / "variations" / "settings" / "settings_validation_method2_smaller_grid_minibatch.yaml"
CNP_TRAIN_CSV = REPO_ROOT / "data/out/cnp/cnp_xlzd_shell_grid75_v1_minibatch_output_15epochs.csv"
CNP_VALIDATION_CSV = REPO_ROOT / "data/out/cnp/cnp_xlzd_shell_grid75_v1_minibatch_output_validation_15epochs.csv"
DATASET_ROOT = REPO_ROOT / "outputs_shell_theta_grid75"

SEED = 42
STEPS_PER_EPOCH = 5000
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
REPR_DIM = 32
HIDDEN = 128
DROPOUT = 0.1
MONITOR_EVERY = 5000
MC_SAMPLES = 30
CHUNK_SIZE = 20000
GRID_POINTS = 120
RANDOM_STATE = 42
TARGET_TRANSFORMS = ["linear", "log_hf", "log_lf", "log_both"]

print(f"Repo root: {REPO_ROOT}")
print(f"Variation: {VARIATION}")
print(f"Prep config: {PREP_CONFIG_PATH}")
print(f"Training config: {CONFIG_PATH}")
print(f"Validation config: {VALIDATION_CONFIG_PATH}")
print(f"Dataset root: {DATASET_ROOT}")


## 1. Inspect The Variation Config


In [ ]:
prep_config = json.loads(PREP_CONFIG_PATH.read_text())
runtime = load_runtime_config(CONFIG_PATH, seed=SEED)
validation_runtime = load_runtime_config(VALIDATION_CONFIG_PATH, seed=SEED)

summary = pd.DataFrame({
    "field": ["version", "dataset_root", "target_mode", "delta_r", "delta_z", "r_shell_step", "z_shell_step", "min_candidate_events", "theta_headers", "phi_headers", "target_headers"],
    "value": [
        runtime.version,
        str(DATASET_ROOT),
        prep_config["shell"].get("target_mode", "hard_box"),
        prep_config["shell"].get("delta_r"),
        prep_config["shell"].get("delta_z"),
        prep_config["shell"].get("r_shell_step"),
        prep_config["shell"].get("z_shell_step"),
        prep_config["shell"].get("min_candidate_events"),
        ", ".join(runtime.theta_headers),
        ", ".join(runtime.phi_headers),
        ", ".join(runtime.target_headers),
    ],
})
summary


## 2. Prepare The Variation Data

Run this once from the repo root before training if the variation dataset has not been built yet.


In [ ]:
PREPARE_COMMAND = "python3 xlzd_shell_theta/variations/run_variation.py --variation method2_smaller_grid --stage prepare_convert"
print(PREPARE_COMMAND)


## 3. Train The CNP


In [ ]:
train_result = train_cnp(
    runtime,
    steps_per_epoch=STEPS_PER_EPOCH,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    repr_dim=REPR_DIM,
    hidden=HIDDEN,
    dropout=DROPOUT,
    monitor_every=MONITOR_EVERY,
    show_monitor_plots=True,
)

pd.DataFrame({
    "artifact": ["model_path", "history_csv", "history_plot", "sample_plot"],
    "path": [str(train_result.model_path), str(train_result.history_csv), str(train_result.history_plot), str(train_result.sample_plot)],
})


## 4. Predict On Training LF/HF And Validation HF


In [ ]:
train_prediction = predict_cnp(
    runtime,
    model_path=train_result.model_path,
    mc_samples=MC_SAMPLES,
    chunk_size=CHUNK_SIZE,
)

validation_prediction = predict_cnp(
    validation_runtime,
    model_path=train_result.model_path,
    mc_samples=MC_SAMPLES,
    chunk_size=CHUNK_SIZE,
)

pd.DataFrame({
    "artifact": [
        "train_csv",
        "train_heatmap",
        "train_error_heatmap",
        "validation_csv",
        "validation_heatmap",
        "validation_error_heatmap",
    ],
    "path": [
        str(train_prediction.csv_path),
        str(train_prediction.heatmap_path),
        str(train_prediction.error_heatmap_path),
        str(validation_prediction.csv_path),
        str(validation_prediction.heatmap_path),
        str(validation_prediction.error_heatmap_path),
    ],
})


## 5. Inspect CNP Outputs


In [ ]:
train_df = pd.read_csv(CNP_TRAIN_CSV)
validation_df = pd.read_csv(CNP_VALIDATION_CSV)

display(train_df.head())
display(validation_df.head())

for path in [
    train_result.history_plot,
    train_result.sample_plot,
    train_prediction.heatmap_path,
    train_prediction.error_heatmap_path,
    validation_prediction.heatmap_path,
    validation_prediction.error_heatmap_path,
]:
    print(path)
    if Path(path).exists():
        try:
            display(Image(filename=str(path)))
        except Exception:
            pass


## 6. Fit The MF-GP Transform Suite


In [ ]:
mfgp_results = run_mfgp_transform_suite(
    config_path=CONFIG_PATH,
    cnp_csv=CNP_TRAIN_CSV,
    validation_csv=CNP_VALIDATION_CSV,
    transforms=TARGET_TRANSFORMS,
    iteration=0,
    grid_points_per_axis=GRID_POINTS,
    random_state=RANDOM_STATE,
    predict_chunk_size=CHUNK_SIZE,
    verbose=True,
)

metrics_rows = []
for transform, result in mfgp_results.items():
    metrics_path = getattr(result, "metrics_json", None)
    if metrics_path and Path(metrics_path).exists():
        payload = json.loads(Path(metrics_path).read_text())
        metrics_rows.append({"transform": transform, **payload})
pd.DataFrame(metrics_rows) if metrics_rows else pd.DataFrame()


## 7. Display MF-GP Plots


In [ ]:
EXPERIMENT_TITLES = {
    "linear": "Normal MF-GP",
    "log_hf": "Log HF: emulate log10(y_raw)",
    "log_lf": "Log LF: use log10(y_cnp)",
    "log_both": "Log HF + Log LF: use log10(y_raw) and log10(y_cnp)",
}

PLOT_SELECTIONS = {
    "linear": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: y with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation 3-sigma: log10(y) with linear sigma", "validation_across_theta_log_linear_sigma_plot"),
        ("Validation parity", "validation_parity_linear_plot"),
    ],
    "log_hf": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: log10 target with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation 3-sigma: y with 10**(mu±sigma)", "validation_across_theta_log_plot"),
        ("Validation parity", "validation_parity_log_plot"),
    ],
    "log_lf": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: y with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation 3-sigma: log10(y) with linear sigma", "validation_across_theta_log_linear_sigma_plot"),
        ("Validation parity", "validation_parity_linear_plot"),
    ],
    "log_both": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: log10 target with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation 3-sigma: y with 10**(mu±sigma)", "validation_across_theta_log_plot"),
        ("Validation parity", "validation_parity_log_plot"),
    ],
}

def show_plot(result, label, key):
    value = getattr(result, key, None)
    if not value:
        return
    print(f"{label}: {value}")
    path = Path(value)
    if path.suffix.lower() == ".png" and path.exists():
        try:
            display(Image(filename=str(path)))
        except Exception:
            pass

for transform in TARGET_TRANSFORMS:
    print(f"\n=== {EXPERIMENT_TITLES.get(transform, transform)} ===")
    result = mfgp_results.get(transform)
    if result is None:
        continue
    for label, key in PLOT_SELECTIONS[transform]:
        show_plot(result, label, key)
